In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

REPO_REV = '4a5963f'
REPO_ROOT = Path('/kaggle/working/spider')
subprocess.run(['git', 'clone', 'https://github.com/yogesh-dhande/spider.git', str(REPO_ROOT)], check=True)
subprocess.run(['git', '-C', str(REPO_ROOT), 'checkout', REPO_REV], check=True)
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT / 'src'))
os.environ['HF_HUB_DOWNLOAD_TIMEOUT'] = '300'
os.environ['HF_HUB_ETAG_TIMEOUT'] = '60'
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'


In [ ]:
%pip install -q --progress-bar off -r requirements/experiment2-kaggle.txt


In [ ]:
from spider.exp4_data import find_exp2_initial_adapter, find_exp4_data
from spider.workflow import gpu_summary

prepared = find_exp4_data('/kaggle/input')
initial_adapter = find_exp2_initial_adapter('/kaggle/input')
os.environ['SPIDER_DATA_DIR'] = str(prepared)
os.environ['SPIDER_INITIAL_ADAPTER'] = str(initial_adapter)
print({'event': 'training_inputs', 'prepared': str(prepared), 'initial_adapter': str(initial_adapter), **gpu_summary()}, flush=True)


In [ ]:
from spider.ddp_smoke import torchrun_command

os.environ['SPIDER_OUTPUT_DIR'] = str(REPO_ROOT / 'outputs/experiment4_compat')
command = torchrun_command('configs/experiment4.yaml', 2, 2, 8, resume='none')
env = os.environ.copy()
env['PYTHONPATH'] = os.pathsep.join(value for value in (str(REPO_ROOT / 'src'), env.get('PYTHONPATH')) if value)
print({'event': 'distributed_training_start', 'command': command}, flush=True)
subprocess.run(command, check=True, env=env)
state = json.loads((REPO_ROOT / 'outputs/experiment4_compat' / 'training_state.json').read_text())
assert state['start_step'] == 0, state
assert state['completed_step'] == 2, state
assert state['planned_epoch_steps'] == 1875, state
assert state['world_size'] == 2, state
assert state['gradient_accumulation_steps'] == 8, state
assert state['effective_batch_size'] == 16, state
print({'event': 'training_stage_complete', 'state': state}, flush=True)


In [ ]:
from spider.action_evaluate import evaluate_actions

adapter = REPO_ROOT / 'outputs/experiment4_compat/adapter/final'
_, metrics = evaluate_actions('configs/experiment4.yaml', 'compat-action', str(adapter), split='development', limit=1)
assert metrics['examples'] == 1, metrics
print({'event': 'initial_adapter_ddp_smoke_complete', 'metrics': metrics}, flush=True)
